# GlyGen Chatbot — End-to-End RAG Pipeline (Global Test)

## Architecture
1. **Ingestion** — PDF → chunk → HuggingFace embed → ChromaDB (`glyco_corpus`)
2. **Query** — embed user question (same model as ingestion)
3. **Retrieval** — similarity search, top **20** chunks
4. **Rerank** — BGE cross-encoder → top **5** chunks
5. **Generation** — GPT-4o with grounded context *(requires OPENAI_API_KEY)*

## Models
| Role | Model |
|------|-------|
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Reranker | `BAAI/bge-reranker-base` |
| Generator | `gpt-4o` *(when API key provided)* |

## Env vars (`.env` at project root)
- `HF_TOKEN` — required for HuggingFace models
- `OPENAI_API_KEY` — optional for now (generation + LLM reranker later)

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / "pyproject.toml").exists():
        PROJECT_ROOT = parent
        break

load_dotenv(PROJECT_ROOT / ".env")

sys.path.insert(0, str(PROJECT_ROOT / "src" / "glygen-chatbot"))

PDF_PATH = PROJECT_ROOT / "Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "glyco_corpus"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "BAAI/bge-reranker-base"
GENERATION_MODEL = "gpt-4o"

TOP_K_RETRIEVAL = 20
TOP_K_RERANK = 5

HF_TOKEN = os.getenv("HF_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Project root:", PROJECT_ROOT)
print("PDF exists:", PDF_PATH.exists())
print("HF_TOKEN set:", bool(HF_TOKEN))
print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))

Project root: D:\Glygen-AI-CHatbot
PDF exists: True
HF_TOKEN set: True
OPENAI_API_KEY set: False


## Phase 1 — Ingestion
Uses existing `Ingestion` package (same as `Ingestion/test.ipynb`).

In [2]:
from Ingestion.config import IngestionConfig
from Ingestion.pipeline import IngestionPipeline

config = IngestionConfig.from_env(start=PROJECT_ROOT)
pipeline = IngestionPipeline(config)

documents = pipeline.load()
chunks = pipeline.chunk(documents)
vectorstore = pipeline.store(chunks)

print(f"Pages loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print(f"Chroma collection: {config.collection_name}")
print(f"Chroma path: {config.chroma_dir}")

c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1433.32it/s]


Pages loaded: 1329
Chunks created: 3656
Chroma collection: glyco_corpus
Chroma path: D:\Glygen-AI-CHatbot\data\chroma


## Phase 2 — Query & Retrieval
Embed the query and fetch top 20 chunks from ChromaDB.

In [3]:
USER_QUERY = "What are glycans?"

embeddings = pipeline.get_embeddings()
vectorstore = pipeline.load_vectorstore()

retrieved_docs = vectorstore.similarity_search(USER_QUERY, k=TOP_K_RETRIEVAL)

print(f"Query: {USER_QUERY}")
print(f"Retrieved: {len(retrieved_docs)} chunks\n")

for i, doc in enumerate(retrieved_docs[:3], start=1):
    print(f"--- Pre-rerank {i} ---")
    print(doc.page_content[:300])
    print(doc.metadata)
    print()

Query: What are glycans?
Retrieved: 20 chunks

--- Pre-rerank 1 ---
sequences. (Reproduced, with permission, from Scientific American, March 2016, p. 76 [Artist: Stephen Smith].
Source: Hinchliff et al. 2015. Proc Natl Acad Sci 112: 12764–12769.)
EVOLUTIONARY VARIATIONS IN GLYCANS
N-Glycans
{'page_label': '422', 'moddate': '2026-06-16T23:55:36+00:00', 'producer': 'ConvertAPI', 'creationdate': '2026-06-16T23:55:27+00:00', 'total_pages': 1329, 'title': 'Essentials of Glycobiology, Fourth Edition', 'creator': '', 'source': 'D:\\Glygen-AI-CHatbot\\Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf', 'author': 'Ajit Varki', 'page': 421}

--- Pre-rerank 2 ---
sequences. (Reproduced, with permission, from Scientific American, March 2016, p. 76 [Artist: Stephen Smith].
Source: Hinchliff et al. 2015. Proc Natl Acad Sci 112: 12764–12769.)
EVOLUTIONARY VARIATIONS IN GLYCANS
N-Glycans
{'creationdate': '2026-06-16T23:55:27+00:00', 'page': 421, 'moddate': '2026-06-16T23:55:36+00:00', 'creator': '', 

## Phase 3 — Reranking (BGE only)
Cross-encoder reranks top 20 → best **5** chunks.

> **Future:** Layer 2 LLM reranker (GPT-4o) when `OPENAI_API_KEY` is provided.

In [4]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

cross_encoder = HuggingFaceCrossEncoder(
    model_name=RERANKER_MODEL,
    model_kwargs={"token": HF_TOKEN},
)

pairs = [(USER_QUERY, doc.page_content) for doc in retrieved_docs]
scores = cross_encoder.score(pairs)

ranked = sorted(zip(retrieved_docs, scores), key=lambda x: x[1], reverse=True)
reranked_docs = [doc for doc, _ in ranked[:TOP_K_RERANK]]

print(f"Reranked to top {TOP_K_RERANK}:\n")
for i, (doc, score) in enumerate(ranked[:TOP_K_RERANK], start=1):
    print(f"--- Rerank {i} (score={score:.4f}) ---")
    print(doc.page_content[:300])
    print(doc.metadata)
    print()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 595.85it/s]


Reranked to top 5:

--- Rerank 1 (score=0.7956) ---
furan.
Galectins: S-type (sulfhydryl-dependent) β-galactoside-binding lectins, usually occurring in
a soluble form, expressed by a wide variety of animal cell types and
distinguishable by the amino acid sequence of their carbohydrate recognition
domains.
Ganglioside: Anionic glycosphingolipid contai
{'moddate': '2026-06-16T23:55:36+00:00', 'author': 'Ajit Varki', 'page': 1172, 'creator': '', 'page_label': '1173', 'source': 'D:\\Glygen-AI-CHatbot\\Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf', 'creationdate': '2026-06-16T23:55:27+00:00', 'total_pages': 1329, 'title': 'Essentials of Glycobiology, Fourth Edition', 'producer': 'ConvertAPI'}

--- Rerank 2 (score=0.7956) ---
furan.
Galectins: S-type (sulfhydryl-dependent) β-galactoside-binding lectins, usually occurring in
a soluble form, expressed by a wide variety of animal cell types and
distinguishable by the amino acid sequence of their carbohydrate recognition
domains.
Gangliosi

## Phase 4 — Generation (GPT-4o)
Runs only when `OPENAI_API_KEY` is set in `.env`.

In [5]:
def build_context(docs):
    return "\n\n".join(
        f"[Source {i} | page={doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for i, doc in enumerate(docs, start=1)
    )

context = build_context(reranked_docs)

if OPENAI_API_KEY:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a glycobiology tutor. Answer ONLY from the provided context. "
            "Cite page numbers. If the context is insufficient, say so.",
        ),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ])

    llm = ChatOpenAI(model=GENERATION_MODEL, temperature=0)
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": USER_QUERY})

    print("=== Generated Answer ===")
    print(response.content)
else:
    print("OPENAI_API_KEY not set — skipping generation.")
    print("Context that would be sent to GPT-4o:\n")
    print(context[:2000])

OPENAI_API_KEY not set — skipping generation.
Context that would be sent to GPT-4o:

[Source 1 | page=1172]
furan.
Galectins: S-type (sulfhydryl-dependent) β-galactoside-binding lectins, usually occurring in
a soluble form, expressed by a wide variety of animal cell types and
distinguishable by the amino acid sequence of their carbohydrate recognition
domains.
Ganglioside: Anionic glycosphingolipid containing one or more residues of sialic acid.
Gene chip: A DNA microarray used to quantify transcript levels in high-throughput format.
Genome: The complete genetic sequence of one set of chromosomes.
Glycan: A generic term for any sugar or assembly of sugars, in free form or attached to
another molecule, used interchangeably in this book with saccharide or
carbohydrate.
Glycan array: A collection of glycans attached to a surface in a spatially determined manner.
Glycan-binding protein: Protein that recognizes and binds to specific glycans. See Lectin and
Glycosaminoglycan-binding protein.

## Future — Layer 2 LLM Reranker (GPT-4o)
When `OPENAI_API_KEY` is available, add a second rerank pass before generation.

In [6]:
if OPENAI_API_KEY:
    print("TODO: GPT-4o reranker — rank reranked_docs again, keep top 5")
else:
    print("LLM reranker skipped — add OPENAI_API_KEY to .env when ready.")

LLM reranker skipped — add OPENAI_API_KEY to .env when ready.
